# 05 — HMM strand: what we tested, what failed, and what replicates

**A results notebook, not a pipeline.** It reads the artifacts already computed by the scripts
and tells the story end-to-end. It runs in seconds (no model fitting here). The live pipeline is
[`01_hmm_triplet.ipynb`](01_hmm_triplet.ipynb); the batch scripts are `scripts/run_hmm_*.py`.

## The question
Decision-making as a **temporal sequence of metastable states**, *shared across areas* but on
*area-specific* geometry. The HMM's job in that claim is narrow and testable: **does the
population pass through a small number of reproducible states, and does it run through them in a
directed sequence — more so in motor midbrain than in visual cortex?**

## The one-paragraph answer
Across **all 5** IBL `{VISp, SCm, MRN}` triplet sessions (5 labs, 40 ms bins, unified
preprocessing): the state count is a **common K = 3**; **both motor areas are more directed than
VISp in every session** (paired sign test *p* = 0.031 each), while **VISp is undirected**
(~chance) throughout. The strong caveat: the analysis window is **mostly movement**, and the
HMM's first evoked transition is **movement-locked**, so this is a *sensory→motor* directedness
gradient, not yet a *decision* sequence.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
from math import comb
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
HMM = ROOT / "results" / "hmm"
FIG = ROOT / "results" / "figures"

def show(relpath, width=920):
    p = ROOT / relpath
    display(Image(filename=str(p), width=width)) if p.exists() else print("missing:", relpath)

print("reading artifacts from", HMM)

## 1. The substrate — 5 sessions, one preprocessing

All five sessions that co-record the whole `{VISp, SCm, MRN}` triplet at ≥ 20 good units, built
through the **single unified pipeline** (`src/data/preprocess.py`): `stimOn` **and**
`firstMovement` alignment, `pre 0.4 / post 0.8 s`, **40 ms bins**, units ≥ 1 Hz, `qc == 1.0`.
The config is serialised inside every `.npz`, so a tensor can't be silently mis-described.

In [ ]:
df = pd.read_csv(HMM / "hmm_sessions.csv")
tbl = (df[df["align"].str.startswith("stimOn")]
       .pivot_table(index=["eid","n_trials"], columns="region", values="n_units")
       .reset_index()[["eid","n_trials","VISp","SCm","MRN"]]
       .sort_values("n_trials", ascending=False))
tbl.columns = ["session","usable trials","VISp units","SCm units","MRN units"]
print("The five triplet sessions (40 ms, units >= 1 Hz):")
display(tbl.reset_index(drop=True))
print("Note the thin populations that will matter later: 90c61c38 SCm=14 units, 90d1e82c MRN=18.")

## 2. Choosing K — the criterion we threw away, and the one we kept

**What failed: held-out log-likelihood.** A Poisson HMM assumes neurons are conditionally
independent given the state; real populations are correlated; the model's *only* way to express
correlation is to add a state. So held-out LL climbs with K forever and its argmax lands on
whichever edge of the search range you look at — it gave **K=2** (bottom, off a column of `-inf`
from a silent-unit bug) and then **K=10** (top) on the *same* data. Two opposite non-answers,
both boundaries. (We first verified the implementation is not at fault: `hmmlearn` matches
`dynamax` on held-out LL to machine precision and on the Viterbi path on 100% of bins —
`scripts/validate_hmm_vs_dynamax.py`.)

**What we kept: state stability (Adjusted Rand Index).** The claim is *reproducible* states, so
measure reproducibility: fit two HMMs on disjoint trial thirds A and B, decode both on a
never-seen third C, and score how much the two independent fits agree (ARI — permutation-
invariant, chance-corrected, comparable across K). Unlike likelihood it has a **peak** and can
say *"no reproducible states here"*. Full argument:
[`ARI_and_state_stability.md`](../docs/knowledge/ARI_and_state_stability.md),
[`choosing_K_for_the_HMM.md`](../docs/knowledge/choosing_K_for_the_HMM.md).

## 3. Is there a common state count? Yes — **K = 3** — and the divergence is *trials*, not biology

We ran the stability analysis **independently on each session** (`run_hmm_kselect.py`) rather than
imposing one session's K on the rest.

In [ ]:
ks = json.load(open(HMM / "k_selection_stimOn_bin40.json"))
rows = []
for region in ["VISp","SCm","MRN"]:
    ps = ks["regions"][region]["per_session"]
    rows.append({"region": region, **{e[:8]: v["best_k"] for e, v in ps.items()}})
best = pd.DataFrame(rows).set_index("region")
print("Per-session BEST (non-trivial) K:")
display(best)
mode = best.mode(axis=1)[0]
print("Modal best-K per region:", dict(mode.astype(int)),
      "\n-> K=3 is the mode in every region.\n")
print("The only high-K argmaxes are 111c1762's VISp=5 / MRN=6 — its 364 trials give flat, noisy\n"
      "ARI curves (low ARI at those K), i.e. a low-power argmax, not a real extra state. See the\n"
      "curves below: the outlier is the one that wanders.")
show("results/figures/phase2_hmm/kselect_stimOn_bin40.png", width=1000)

**Verdict (`common_K_across_sessions.md`): K = 3 for all three regions, defensible** — every
session sits within fold-noise of K = 3 in SCm and MRN, and 4/5 do in VISp (the 5th being the
low-trial outlier). SCm additionally supports K = 4 in the best-powered session (5c0c560e), but
K = 3 is within noise even there. So the replication is fit at **K = 3 everywhere**; since
`forward_fraction` is flat across K, the directedness result below does not depend on 3-vs-4.

## 4. The result that replicates — motor midbrain is directed, visual cortex is not

`forward_fraction` = share of state transitions running to a *later*-ordered state (0.5 = symmetric
mixing, →1 = a clean forward chain), computed **split-half** so the state ordering is derived on
held-out trials (closes the ordering-then-testing-the-ordering circularity). One value per session.

In [ ]:
piv = df.pivot(index="eid", columns="region", values="fwd_frac_split_half")[["VISp","SCm","MRN"]]
print("forward_fraction (split-half), per session:")
display(piv.round(3))

pooled = pd.read_csv(HMM / "hmm_sessions_pooled.csv").set_index("region")
print("Pooled across 5 sessions (mean +/- SD):")
for r in ["VISp","SCm","MRN"]:
    print(f"  {r}: {pooled.loc[r,'fwd_split_mean']:.3f} +/- {pooled.loc[r,'fwd_split_sd']:.3f}")

print("\nPAIRED sign test — is each motor area more directed than VISp, session by session?")
for reg in ["SCm","MRN"]:
    d = (piv[reg] - piv["VISp"]).to_numpy()
    npos, n = int((d > 0).sum()), len(d)
    p = sum(comb(n,k) for k in range(npos, n+1)) / 2**n
    print(f"  {reg} > VISp in {npos}/{n} sessions  (mean +{d.mean():.3f})  sign-test p = {p:.3f}")
print(f"\n  VISp vs chance: {piv['VISp'].mean():.3f} +/- {piv['VISp'].std():.3f}  (~0.5 = undirected)")
show("results/figures/phase2_hmm/sessions_forward_fraction.png", width=680)

**Read-out.** SCm > VISp and MRN > VISp in **5/5 sessions each** (sign test *p* = 0.031),
across 5 labs and 5 mice — a genuine animal-independent replication, stronger than one session
with tighter error bars. **VISp is undirected** (0.54 ± 0.04, ~chance) in every session.

**Honesty about effect size.** The *direction* is unanimous; the *magnitude* is session-dependent
and tracks statistical power — SCm's directedness is weakest in `90c61c38` (only **14** SCm units)
and MRN's in `90d1e82c` (only **18** MRN units). So the gradient is real and replicable, but its
size depends on having enough simultaneously-recorded units. This is the "trials/units, not the
brain" limit the whole project keeps meeting.

One HMM fit, to see what "directed" looks like — SCm K=3-4, `5c0c560e`: baseline → a transient
state → a sustained state, with the net-flux almost entirely in the forward (upper) triangle:

In [ ]:
show("results/figures/phase2_hmm/SCm_hmm.png", width=1000)

## 5. VISp is *undirected*, not *static* — a dissociation worth keeping

A tempting misreading of "VISp ≈ 0.5" is "VISp barely changes state". The opposite is true:
`forward_fraction` is an **asymmetry**, `switch_rate` is a **frequency**, and they dissociate.

In [ ]:
sw = df.pivot(index="eid", columns="region", values="switch_rate")[["VISp","SCm","MRN"]]
print("switch_rate (state changes / trial), per session:")
display(sw.round(2))
print(f"VISp mean switch_rate = {sw['VISp'].mean():.2f}/trial — it switches as much as (often MORE")
print("than) the motor areas. It simply switches with NO preferred direction. Undirected != static.")

## 6. The load-bearing caveat — the window is mostly *movement*

Median reaction time is **171 ms** and median feedback **427 ms**, so over `[0,+0.8]s` movement
has begun on ~92% of trials. We tested whether the HMM's first evoked transition is locked to the
**stimulus** or to the **wheel-turn**, by fitting on both alignments and comparing the
across-trial jitter of the state-exit time (`run_movement_alignment.py`). A movement-locked
transition gets *sharper* when you align to movement; a stimulus-locked one is smeared by the RT
jitter.

In [ ]:
mv = pd.read_csv(HMM / "movement_alignment.csv")
print("Exit-time jitter (SD across trials), 5c0c560e, 40 ms:")
display(mv[["region","sd_exit_stimOn_ms","sd_exit_move_ms","sd_rt_ms","tighter_to"]])
print("Every region's transition is tighter to MOVEMENT than to the stimulus (44-153 ms aligned")
print("to movement, vs ~350 ms expected if it were stimulus-locked). The 'evoked' state IS,")
print("to first order, a movement state.")
show("results/figures/phase2_hmm/5c0c560e_occupancy_stim_vs_move.png", width=1000)

In the movement frame (bottom row) the baseline collapse and evoked-state rise **snap to
t = 0** and peak higher than in the stimulus frame (top). **Consequence: no state after ~0 may be
called a "decision" state.** The directedness gradient in §4 is, as measured, a *sensory → motor
execution* gradient. Separating decision from movement needs the movement-aligned replication
(the tensors are built; the per-session movement analysis is the next run).

## 7. The ledger — what we tested, and where it landed

| tested | outcome |
|---|---|
| held-out likelihood to pick K | ✗ **rejected** — climbs forever; argmax hit both boundaries (K=2 off `-inf`, then K=10) |
| our HMM vs `dynamax` | ✓ machine-precision agreement — the implementation is sound |
| **state stability (ARI)** for K | ✓ **adopted** — has a peak, chance-corrected, can say "no" |
| K across 5 sessions | ✓ **common K = 3**; apparent divergence is the lowest-trial session's noise |
| **directedness gradient (motor > visual)** | ✓ **replicates 5/5**, sign test *p* = 0.031; VISp undirected |
| state-count gradient (VISp<MRN<SCm) | ✗ **retracted** — an EM-restart artifact; counts are 3/3/3 |
| MRN directedness magnitude | ⚠ **weakened** by the split-half test (ordering circularity) and thin-unit sessions |
| latency cascade VISp→SCm→MRN | ⚠ real in well-powered sessions, but **confounded with RT/feedback** |
| is the evoked state a decision or a movement | ⚠ **movement** — transition is movement-locked |

**What we would put in a talk:** *across five mice and five labs, motor midbrain traverses its
states more directedly than visual cortex (5/5, p = 0.031), on a shared count of ~3 reproducible
states — but at this window the directed transition is movement-locked, so it is a sensory→motor
gradient until the movement-aligned control separates decision from action.*

## 8. Limitations
1. **Movement confound** (§6) — the biggest one; the movement-aligned replication is the next step.
2. **Effect size is power-limited** — thin-unit sessions weaken the gradient (§4).
3. **`stimOn` only** here — the whole analysis should be repeated on the `firstMovement` tensors.
4. **Poisson conditional-independence** is violated (correlations), which is *why* likelihood fails
   for K and part of *why* the rSLDS (continuous latent) is the principled next model.

## Reproduce
```
python scripts/build_processed_all.py --bin-ms 40           # all 5 sessions, both alignments
python scripts/run_hmm_kselect.py    --bin-ms 40 --align stimOn      # per-session K + common-K
python scripts/run_hmm_sessions.py   --bin-ms 40 --align stimOn --common-k 3   # the replication
python scripts/run_movement_alignment.py --bin-ms 40        # the movement test
```